In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec


In [ ]:
import os
import pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pc.create_index(
    name="profspot-csuf", dimension=1536, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

In [ ]:
import json
data = json.load(open("reviews.json"))
data['reviews']

In [ ]:
from openai import OpenAI
processed_data = []
client = OpenAI()

for review in data['reviews']:
    response = client.embeddings.create(
        input=review['review'],
        model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding
    processed_data.append({
        "values": embedding,
        "id": f"{review['professor']} | {review['subject']}",
        "metadata": {
            "review": review["review"],
            "subject": review["subject"],
            "stars": review["stars"]

        }
    })

In [ ]:
processed_data[0]

In [ ]:
index = pc.Index('profspot-csuf')
index.upsert(
    vectors=processed_data,
    namespace="csuf"
    
)

In [ ]:
index.describe_index_stats()